# Elastic MCP Multi-agent Demo

## Create Elastic Serverless Environment

In [1]:
%%bash
terraform -chdir=terraform init  -upgrade
terraform -chdir=terraform apply -auto-approve

Initializing the backend...
Initializing provider plugins...
- Finding elastic/elasticstack versions matching "~> 0.14"...
- Finding elastic/ec versions matching "~> 0.12"...
- Using previously-installed elastic/elasticstack v0.14.3
- Using previously-installed elastic/ec v0.12.5

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be created
  + resource "ec_elastics

## Create Environment File

In [2]:
%%bash
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
KIBANA_URL=$(terraform -chdir=terraform output -raw kibana_url)
GOOGLE_CLOUD_PROJECT=$(terraform -chdir=terraform output -raw google_cloud_project)
GOOGLE_CLOUD_LOCATION=$(terraform -chdir=terraform output -raw google_cloud_location)
OPENAI_API_KEY=$(terraform -chdir=terraform output -raw openai_api_key)
ANTHROPIC_API_KEY=$(terraform -chdir=terraform output -raw anthropic_api_key)
EOF

## Install Requirements

In [3]:
! pip install -q -U -r requirements.txt

## Index Data

In [4]:
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from elasticsearch import helpers
import json
from tqdm.notebook import tqdm

load_dotenv(override=True)

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"]
)

PRODUCT_INDEX = "products"
REVIEW_INDEX = "reviews"
products_mapping = {
    "mappings": {
        "_meta": {
            "description": "Index of products available in our catalog"
        },
        "properties": {
            "product_id":     {"type": "keyword"},
            "name":           {"type": "text"},
            "description":    {
                "type":     "text",
                "fields":      {"embedding": {"type": "semantic_text"}}
            },
            "category":     {"type": "keyword"},
            "price":        {"type": "float"},
            "stock_count":  {"type": "integer"},
        }
    }
}
reviews_mapping = {
    "mappings": {
        "_meta": {
            "description": "Index of customer reviews for products"
        },
        "properties": {
            "product_id":    {"type": "keyword"},
            "review":   {
                "type":     "text",
                "fields":   {"embedding": {"type": "semantic_text"}}
            }, 
            "stars":        {"type": "integer"},
            "review_date":  {"type": "date"}
        }
    }
}
es.options(ignore_status=[404]).indices.delete(index=PRODUCT_INDEX)
es.options(ignore_status=[404]).indices.delete(index=REVIEW_INDEX)
es.indices.create(index=PRODUCT_INDEX, body=products_mapping)
es.indices.create(index=REVIEW_INDEX, body=reviews_mapping)

def gen_data(data, index_name):
    for doc in tqdm(data, desc=f"Indexing {index_name}"):
        yield { "_index": index_name, "_source": doc}

with open("assets/products.jsonl") as f:
    products = [json.loads(line) for line in f]
with open("assets/reviews.jsonl") as f:
    reviews = [json.loads(line) for line in f]

ok_products, errors = helpers.bulk(es, gen_data(products, PRODUCT_INDEX))
ok_reviews, errors = helpers.bulk(es, gen_data(reviews, REVIEW_INDEX))
print(f"Successfully indexed {ok_reviews} reviews and {ok_products} products")

Indexing products:   0%|          | 0/100 [00:00<?, ?it/s]

Indexing reviews:   0%|          | 0/393 [00:00<?, ?it/s]

Successfully indexed 393 reviews and 100 products


## Create Elastic Tools

In [5]:
import requests
import os

ENDPOINT = "/api/agent_builder/tools"

TOOL_1_ID = "search_products"
TOOL_2_ID = "get_reviews"

headers = {
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
    "Authorization": f"ApiKey {os.environ.get('ELASTIC_CLOUD_API_KEY')}"
}

esql_1 = '''
FROM products METADATA _id, _index, _score
| FORK
  (WHERE MATCH(name, ?search_term, {"boost": 2.0}) OR MATCH(description, ?search_term) | SORT _score DESC | LIMIT 100)
  (WHERE MATCH(description.embedding, ?search_term) | SORT _score DESC | LIMIT 100)
| FUSE LINEAR WITH { "normalizer": "minmax" }
| KEEP product_id, name, description, price, _score
| RERANK ?search_term ON description WITH { "inference_id": ".jina-reranker-v3" }
| SORT _score DESC
| LIMIT 3
'''
esql_1 = " ".join(esql_1.replace('\n', ' ').split())

esql_2 = '''
FROM reviews
| WHERE product_id == ?id
| KEEP review
| LIMIT 5
'''
esql_2 = " ".join(esql_2.replace('\n', ' ').split())

try:
    requests.delete(
        f"{os.environ.get('KIBANA_URL')}{ENDPOINT}/{TOOL_1_ID}",
        headers=headers
    )
    requests.delete(
        f"{os.environ.get('KIBANA_URL')}{ENDPOINT}/{TOOL_2_ID}",
        headers=headers
    )

    response = requests.post(
        f"{os.environ.get('KIBANA_URL')}{ENDPOINT}",
        headers=headers,
        json={
            "id": TOOL_1_ID,
            "tags": ["office_products"],
            "type": "esql",
            "description": "Search tool for finding products that best match the natural language query provided by the user.",
            "configuration": {
                "query": esql_1,
                "params": {
                    "search_term": {
                        "type": "string",
                        "description": "The natural language query provided by the user to search for relevant products."
                    }
                }
            }
        }
    )
    response.raise_for_status()

    response = requests.post(
        f"{os.environ.get('KIBANA_URL')}{ENDPOINT}",
        headers=headers,
        json={
            "id": TOOL_2_ID,
            "tags": ["office_products"],
            "type": "esql",
            "description": "Search tool for finding reviews for a given product id.",
            "configuration": {
                "query": esql_2,
                "params": {
                    "id": {
                        "type": "string",
                        "description": "The product ID for which to retrieve reviews."
                    }
                }
            }
        }
    )
    response.raise_for_status()
    print(f"Tools {TOOL_1_ID} and {TOOL_2_ID} created successfully.")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

Tools search_products and get_reviews created successfully.


## Agent Scenario 1 - Google ADK on Vertex.ai

In [6]:
import os
import vertexai
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from vertexai import agent_engines
from google.cloud import storage

GEMINI_MODEL = "gemini-2.5-flash"
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION")
BUCKET = f"gs://{PROJECT}-adk-staging"

vertexai.init(
    project=PROJECT,
    location=LOCATION,
    staging_bucket=BUCKET
)

# --- Local tool: mock shipping calculator ---

def calculate_shipping() -> dict:
    """Calculate estimated shipping cost and delivery date for a given zip code."""
    return {
        "shipping_cost": "$8.99",
        "estimated_delivery": "3-5 business days"
    }

# --- Remote tools: Elastic MCP (search_products + get_reviews) ---

elastic_mcp_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"{os.environ['KIBANA_URL']}/api/agent_builder/mcp",
        headers={
            "Authorization": f"ApiKey {os.environ['ELASTIC_CLOUD_API_KEY']}",
            "Content-Type": "application/json",
        },
    )
)

# --- Assemble the ADK agent ---
agent = LlmAgent(
    name="product_research_agent",
    model=GEMINI_MODEL,
    description="An AI product researcher that searches products and reviews using Elasticsearch.",
    instruction=(
        "You are a helpful product research assistant with access to an Elasticsearch "
        "product catalog and customer reviews.\n\n"
        "When a user asks about products:\n"
        "1. Use search_products to find relevant products matching their query.\n"
        "2. Use get_reviews with the product_id from results to get customer feedback.\n"
        "3. Use calculate_shipping if the user asks about shipping.\n"
        "4. Synthesize product information, reviews, and shipping into a helpful response.\n\n"
        "Always explain your reasoning as you work through the steps."
    ),
    tools=[elastic_mcp_tools, calculate_shipping],
)

# --- Deploy to Vertex AI Agent Engine ---

remote_agent = agent_engines.create(
    agent_engine=agent,
    requirements=[
        "google-adk",
        "google-cloud-aiplatform",
        "google-genai",
        "mcp",
        "httpx",
        "cloudpickle", 
        "pydantic"
    ],
)
print(f"Deployed: {remote_agent.resource_name}")

# --- Query the deployed agent ---

async for event in remote_agent.async_stream_query(
    user_id="demo_user",
    message=(
        "I'm looking for a standing desk that can fit dual monitors for my home office. "
        "Find me a good option in our catalog, and then summarize what actual customer reviews say about stability and motor noise. "
        "Include estimated shipping cost and delivery time as well."
    ),
):
    parts = (event.get("content") or {}).get("parts") or []
    for part in parts:
        if part.get("functionCall"):
            fc = part["functionCall"]
            print(f"\n🔧 Tool Call: {fc['name']}")
            print(f"   Args: {fc.get('args', {})}")
        elif part.get("functionResponse"):
            fr = part["functionResponse"]
            resp = str(fr.get("response", ""))
            print(f"\n📋 Tool Response ({fr['name']}): {resp[:500]}{'...' if len(resp) > 500 else ''}")
        elif part.get("text"):
            has_tool_parts = any(p.get("functionCall") or p.get("functionResponse") for p in parts)
            if not has_tool_parts and not event.get("partial"):
                print("\n" + "=" * 80)
                print("Agent Response:\n")
                print(part["text"])

# Cleanup: delete the remote agent and staging bucket
print("\n" + "=" * 80)
remote_agent.delete(force=True)
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET.removeprefix("gs://"))
bucket.delete(force=True)

/home/joey/Dev/elastic-mcp-multi-agent-demo/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey
/home/joey/Dev/elastic-mcp-multi-agent-demo/.venv/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


Deploying google.adk.agents.Agent as an application.
Identified the following requirements: {'google-cloud-aiplatform': '1.148.1', 'pydantic': '2.13.2', 'cloudpickle': '3.1.2'}
The final list of requirements: ['google-adk', 'google-cloud-aiplatform', 'google-genai', 'mcp', 'httpx', 'cloudpickle', 'pydantic']
Creating bucket elastic-customer-eng-adk-staging in location='us-central1'
Wrote to gs://elastic-customer-eng-adk-staging/agent_engine/agent_engine.pkl
Writing to gs://elastic-customer-eng-adk-staging/agent_engine/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://elastic-customer-eng-adk-staging/agent_engine/dependencies.tar.gz
Creating AgentEngine
Create AgentEngine backing LRO: projects/881345020217/locations/us-central1/reasoningEngines/2045646331279572992/operations/6510442174671224832
View progress and logs at https://console.cloud.google.com/logs/query?project=elastic-customer-eng
AgentEngine created. Resource name: projects/881345020217/locations/

## Agent Scenario 2 - LangGraph

In [7]:
import os
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

OPENAI_MODEL = "gpt-5.4-nano"

load_dotenv(override=True)

# --- Local tool: mock shipping calculator ---

@tool
def calculate_shipping() -> dict:
    """Calculate estimated shipping cost and delivery date for a given zip code."""
    return {"shipping_cost": "$8.99", "estimated_delivery": "3-5 business days"}

# --- LLM ---

model = ChatOpenAI(model=OPENAI_MODEL, api_key=os.environ["OPENAI_API_KEY"])

# --- MCP connection + Agent ---

client = MultiServerMCPClient(
    {
        "elasticsearch": {
            "transport": "streamable_http",
            "url": f"{os.environ['KIBANA_URL']}/api/agent_builder/mcp",
            "headers": {
                "Authorization": f"ApiKey {os.environ['ELASTIC_CLOUD_API_KEY']}",
                "Content-Type": "application/json",
            },
        }
    }
)
mcp_tools = await client.get_tools()

agent = create_agent(
    model,
    tools=mcp_tools + [calculate_shipping],
    system_prompt=(
        "You are a helpful product research assistant with access to an Elasticsearch "
        "product catalog and customer reviews.\n\n"
        "When a user asks about products:\n"
        "1. Use search_products to find relevant products matching their query.\n"
        "2. Use get_reviews with the product_id from results to get customer feedback.\n"
        "3. Use calculate_shipping if the user asks about shipping.\n"
        "4. Synthesize product information, reviews, and shipping into a helpful response.\n\n"
        "Always explain your reasoning as you work through the steps."
    ),
)

# --- Stream agent execution ---

async for event in agent.astream_events(
    {"messages": [("user",
        "I need a quiet mechanical keyboard for programming in a shared office. "
        "Find me a good option and summarize what customers say about the typing feel and noise level. "
        "What's the estimated shipping?"
    )]},
    version="v2",
):
    kind = event["event"]
    if kind == "on_tool_start":
        print(f"\n🔧 Tool Call: {event['name']}")
        print(f"   Args: {event['data'].get('input', {})}")
    elif kind == "on_tool_end":
        output = str(event["data"].get("output", ""))
        print(f"\n📋 Tool Response ({event['name']}): {output[:500]}{'...' if len(output) > 500 else ''}")
    elif kind == "on_chat_model_end":
        msg = event["data"]["output"]
        if hasattr(msg, "content") and msg.content and not getattr(msg, "tool_calls", None):
            print("\n" + "=" * 80)
            print("Agent Response:\n")
            print(msg.content)


🔧 Tool Call: search_products
   Args: {'search_term': 'quiet mechanical keyboard for programming in office low noise quiet switches'}

📋 Tool Response (search_products): content=[{'type': 'text', 'text': '{"results":[{"type":"query","data":{"esql":"FROM products METADATA _id, _index, _score\\n| FORK\\n      (\\n          WHERE\\n        MATCH(\\n          name,\\n          \\"quiet mechanical keyboard for programming in office low noise quiet switches\\",\\n          {\\"boost\\": 2.0}) OR\\n          MATCH(\\n            description,\\n            \\"quiet mechanical keyboard for programming in office low noise quiet switches\\")\\n      | SORT _score DESC\\n  ...

🔧 Tool Call: get_reviews
   Args: {'id': 'KEYB-0047'}

🔧 Tool Call: get_reviews
   Args: {'id': 'KEYB-0042'}

🔧 Tool Call: get_reviews
   Args: {'id': 'KEYB-0040'}

🔧 Tool Call: calculate_shipping
   Args: {}

📋 Tool Response (calculate_shipping): content='{"shipping_cost": "$8.99", "estimated_delivery": "3-5 business days

## Agent Scenario 3 - CrewAI

In [8]:
import os
import warnings
os.environ["CREWAI_TESTING"] = "true"
warnings.filterwarnings("ignore", message="coroutine.*was never awaited", category=RuntimeWarning)

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.mcp import MCPServerHTTP
from crewai.tools import tool
from crewai.agents.parser import AgentAction

ANTHROPIC_MODEL = "anthropic/claude-haiku-4-5-20251001"

load_dotenv(override=True)

# --- Local tool: mock shipping calculator ---

@tool("calculate_shipping")
def calculate_shipping() -> dict:
    """Calculate estimated shipping cost and delivery date for a given zip code."""
    return {"shipping_cost": "$8.99", "estimated_delivery": "3-5 business days"}

# --- Step callback: print only tool calls ---

def on_step(step):
    if isinstance(step, AgentAction):
        print(f"\n🔧 Tool Call: {step.tool}")
        print(f"   Args: {step.tool_input}")
        if step.result:
            result = str(step.result)
            print(f"\n📋 Tool Response ({step.tool}): {result[:500]}{'...' if len(result) > 500 else ''}")

# --- LLM ---

llm = LLM(
    model=ANTHROPIC_MODEL,
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=4096,
)

# --- MCP connection via Agent ---

agent = Agent(
    role="Product Research Assistant",
    goal=(
        "Search the product catalog, retrieve customer reviews, "
        "and provide shipping estimates to help users make informed purchasing decisions."
    ),
    backstory=(
        "You are a helpful product research assistant with access to an Elasticsearch "
        "product catalog and customer reviews. When a user asks about products: "
        "1. Use search_products to find relevant products. "
        "2. Use get_reviews with the product_id to get customer feedback. "
        "3. Use calculate_shipping if the user asks about shipping. "
        "4. Synthesize everything into a helpful response."
    ),
    tools=[calculate_shipping],
    mcps=[
        MCPServerHTTP(
            url=f"{os.environ['KIBANA_URL']}/api/agent_builder/mcp",
            headers={
                "Authorization": f"ApiKey {os.environ['ELASTIC_CLOUD_API_KEY']}",
                "Content-Type": "application/json",
            },
            streamable=True,
        )
    ],
    llm=llm,
    verbose=False,
    step_callback=on_step,
)

# --- Task ---

task = Task(
    description=(
        "I need a high-resolution monitor with accurate colors for design work and coding. "
        "Find me a good option and summarize what customers say about color accuracy and eye strain. "
        "What's the estimated shipping?"
    ),
    expected_output=(
        "A product recommendation with name, price, and product ID, "
        "a summary of customer reviews about color accuracy and eye strain, "
        "and estimated shipping cost and delivery time."
    ),
    agent=agent,
)

# --- Crew ---

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()

print("\n" + "=" * 80)
print("Agent Response:\n")
print(result.raw)


🔧 Tool Call: demoproject-bd0a1c_kb_us-central1_gcp_elastic_cloud_api_agent_builder_mcp_search_products
   Args: {"search_term": "high-resolution monitor accurate colors design coding"}

📋 Tool Response (demoproject-bd0a1c_kb_us-central1_gcp_elastic_cloud_api_agent_builder_mcp_search_products): {"results":[{"type":"query","data":{"esql":"FROM products METADATA _id, _index, _score\n| FORK\n      (\n          WHERE\n        MATCH(name, \"high-resolution monitor accurate colors design coding\",\n          {\"boost\": 2.0}) OR\n          MATCH(\n            description,\n            \"high-resolution monitor accurate colors design coding\")\n      | SORT _score DESC\n      | LIMIT 100\n      )\n      (\n          WHERE\n        MATCH(\n          description.embedding,\n          \"high-res...

🔧 Tool Call: demoproject-bd0a1c_kb_us-central1_gcp_elastic_cloud_api_agent_builder_mcp_get_reviews
   Args: {"id": "MONI-0081"}

📋 Tool Response (demoproject-bd0a1c_kb_us-central1_gcp_elastic_cloud_a

## Destroy Environment

In [9]:
%%bash
terraform -chdir=terraform destroy -auto-approve
rm -f .env

ec_elasticsearch_project.demo_project: Refreshing state... [id=bd0a1c5251314ef6b264a14f911d29ca]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be destroyed
  - resource "ec_elasticsearch_project" "demo_project" {
      - alias         = "demoproject" -> null
      - cloud_id      = "demo_project:dXMtY2VudHJhbDEuZ2NwLmVsYXN0aWMuY2xvdWQkYmQwYTFjNTI1MTMxNGVmNmIyNjRhMTRmOTExZDI5Y2EuZXMkYmQwYTFjNTI1MTMxNGVmNmIyNjRhMTRmOTExZDI5Y2Eua2I=" -> null
      - credentials   = {
          - password = (sensitive value) -> null
          - username = "admin" -> null
        } -> null
      - endpoints     = {
          - elasticsearch = "https://demoproject-bd0a1c.es.us-central1.gcp.elastic.cloud" -> null
          - kibana        = "https://demoproject-bd0a1c.kb.us-central1.gcp.elastic.cloud" -> null
  